# ecommerce_analytics_demo — Pipeline Walkthrough

**Multi-table enrichment, margin analysis, and temporal trends.**

| Layer | What happens |
|-------|--------------|
| Bronze | Reads 3 source CSVs → `data/bronze/{orders,products,customers}.parquet` |
| Silver (phase 1) | Type-casts each bronze table |
| Silver (phase 2) | UDF joins all three → `order_lines_enriched.parquet` |
| Gold | 3 aggregations (revenue by category, top customers, monthly trend) |

Run cells top to bottom. A single `medallion run ecommerce` command runs the full pipeline.

## Setup — navigate to example root

In [1]:
import os
from pathlib import Path
import polars as pl

# Two levels up: ipynb/ → ecommerce/ → ecommerce_analytics_demo/
example_root = Path(os.getcwd()).parent.parent
os.chdir(example_root)
print(f'Working directory: {os.getcwd()}')

Working directory: /workspace/openmedallion/examples/ecommerce_analytics_demo


## Source data

Three CSV files in `data/source/` — column names are already lowercase.

In [2]:
src = Path('data/source')
print('── orders (first 4) ──')
print(pl.read_csv(src / 'orders.csv').head(4))
print('\n── products ──')
print(pl.read_csv(src / 'products.csv'))
print('\n── customers ──')
print(pl.read_csv(src / 'customers.csv'))

── orders (first 4) ──
shape: (4, 6)
┌──────────┬────────────┬─────────────┬─────┬────────────┬───────────┐
│ order_id ┆ product_id ┆ customer_id ┆ qty ┆ order_date ┆ status    │
│ ---      ┆ ---        ┆ ---         ┆ --- ┆ ---        ┆ ---       │
│ i64      ┆ i64        ┆ i64         ┆ i64 ┆ str        ┆ str       │
╞══════════╪════════════╪═════════════╪═════╪════════════╪═══════════╡
│ 1        ┆ 1          ┆ 1           ┆ 1   ┆ 2024-01-05 ┆ completed │
│ 2        ┆ 2          ┆ 1           ┆ 2   ┆ 2024-01-12 ┆ completed │
│ 3        ┆ 4          ┆ 3           ┆ 2   ┆ 2024-01-18 ┆ completed │
│ 4        ┆ 7          ┆ 2           ┆ 3   ┆ 2024-01-25 ┆ completed │
└──────────┴────────────┴─────────────┴─────┴────────────┴───────────┘

── products ──
shape: (8, 5)
┌────────────┬───────────────────────┬─────────────┬────────────┬───────────┐
│ product_id ┆ product_name          ┆ category    ┆ unit_price ┆ unit_cost │
│ ---        ┆ ---                   ┆ ---         ┆ ---        ┆ -

## Run the full pipeline

`medallion run ecommerce` runs Bronze → Silver → Gold in one shot.

- **Bronze** reads the 3 source CSVs and writes them to `data/bronze/`
- **Silver** casts types and joins tables via a UDF into `order_lines_enriched`
- **Gold** aggregates into 3 output tables

In [3]:
!medallion run ecommerce --layer bronze

📋  [config] loaded bronze: ecommerce/backend/bronze.yaml
📋  [config] loaded silver: ecommerce/backend/silver.yaml
📋  [config] loaded gold  : ecommerce/backend/gold.yaml
Traceback (most recent call last):
  File "/opt/venv/bin/medallion", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/openmedallion/cli/main.py", line 231, in main
    _HANDLERS[_args.command](_args)
  File "/opt/venv/lib/python3.12/site-packages/openmedallion/cli/main.py", line 77, in cmd_run
    cfg = load_project(args.project, args.projects)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/openmedallion/config/loader.py", line 149, in load_project
    _validate_config(cfg)
  File "/opt/venv/lib/python3.12/site-packages/openmedallion/config/validator.py", line 46, in _validate_config
    require(
  File "/opt/venv/lib/python3.12/site-packages/openmedallion/config/validator.py", line 26, in require
    raise Va

## Inspect bronze output

Bronze wrote parquet files directly from the source CSVs.

In [ ]:
print('── data/bronze/orders.parquet (first 4) ──')
print(pl.read_parquet('data/bronze/orders.parquet').head(4))
print('\n── data/bronze/products.parquet ──')
print(pl.read_parquet('data/bronze/products.parquet'))
print('\n── data/bronze/customers.parquet ──')
print(pl.read_parquet('data/bronze/customers.parquet'))

## Inspect silver — enriched join table

`build_order_lines_enriched()` joins orders + products + customers and adds margin columns.

In [ ]:
enriched = pl.read_parquet('data/silver/order_lines_enriched.parquet')
print(f'Enriched table: {enriched.shape[0]} rows × {enriched.shape[1]} cols')
display_cols = ['order_id', 'name', 'product_name', 'category', 'qty',
                'unit_price', 'line_revenue', 'margin_amount']
print(enriched.select(display_cols).sort('order_id').head(6))

## Gold outputs

All aggregations run on `order_lines_enriched.parquet`:

| Output | Group by | Metrics |
|--------|----------|---------|
| `revenue_by_category` | category | total_revenue, total_margin, num_orders |
| `top_customers` | customer_id, name, region, tier | total_spent, num_orders |
| `monthly_summary` | order_month (via pre_agg_udf) | monthly_revenue, num_orders |

In [ ]:
gold = Path('data/gold/ecommerce')

print('── Revenue by Category ──')
cat = pl.read_parquet(gold / 'revenue_by_category.parquet').sort('total_revenue', descending=True)
cat = cat.with_columns(
    (pl.col('total_margin') / pl.col('total_revenue') * 100).round(1).alias('margin_pct')
)
print(cat)

In [ ]:
print('── Top Customers (by spend) ──')
cust = pl.read_parquet(gold / 'top_customers.parquet').sort('total_spent', descending=True)
print(cust)

gold_tier_rev = cust.filter(pl.col('tier') == 'gold')['total_spent'].sum()
total_rev     = cust['total_spent'].sum()
print(f'\nGold-tier revenue share: {gold_tier_rev / total_rev * 100:.1f}%')

In [ ]:
print('── Monthly Revenue Trend ──')
monthly = pl.read_parquet(gold / 'monthly_summary.parquet').sort('order_month')
print(monthly)

peak = monthly.sort('monthly_revenue', descending=True).row(0, named=True)
print(f"\nPeak month: {peak['order_month']}  (${peak['monthly_revenue']:,.2f})")

## Things to Try

- **Add margin_pct**: compute `total_margin / total_revenue` in `add_metrics()` and `backend/gold.yaml`
- **Filter cancelled orders**: add a `filter` transform in `backend/silver.yaml` for `status != 'cancelled'`
- **Revenue by region**: add a new aggregation to `backend/gold.yaml` grouping by `region`